# AI CUP 2026 圍棋 — 競賽二棋力預測訓練（Kaggle Notebooks）

## 使用前務必先做的三件事

1. 右側 **Settings → Accelerator → GPU P100**（或 T4 x2）
2. 右側 **Settings → Internet → On**（需要先完成手機驗證，否則無法 clone / 下載資料）
3. 長訓練請用右上角 **Save Version → Save & Run All (Commit)**，
   這樣會在背景 headless 跑，上限 12 小時；直接在網頁上跑會因為閒置被中斷

## 為什麼只訓練競賽二

競賽一（玩家辨識）用純 CPU 的手工特徵已達 **0.6726**，是官方 baseline（0.2845）的 2.36 倍，
不需要 GPU。競賽二的手工特徵在 0.3973 觸頂、低於官方 baseline 的 0.4631，
必須靠深度模型——GPU 只為這一題而用。

## 1. 環境檢查

In [ ]:
import subprocess, torch, os, multiprocessing
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('CPU 核心', multiprocessing.cpu_count())
print('可寫目錄 /kaggle/working:', os.access('/kaggle/working', os.W_OK))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || true

## 2. 取得程式碼與資料

不需要任何 `pip install`：訓練路徑只用到 numpy / pandas / sklearn / torch，
Kaggle 全部預裝（本專案的 SGF parser 是自己寫的，不依賴 sgfmill）。

In [ ]:
%cd /kaggle/working
!test -d AI-CUP-2026-GO || git clone -q https://github.com/ShiYu0318/AI-CUP-2026-GO.git
%cd /kaggle/working/AI-CUP-2026-GO
!git pull -q 2>/dev/null || true
!ls

In [ ]:
# 官方訓練集 214 MB，直接從 GitHub Releases 下載（不要從本機上傳）
%cd /kaggle/working/AI-CUP-2026-GO
URL = 'https://github.com/AILAB-NDHU/aicup2026-go-dataset/releases/download/v1.0/aicup2026_go_training.tar.xz'
SHA = '959ebc602b9638d665cb063a6269e0d5aef73956b77913e725763eb335ced94b'
!test -d training || wget -q --show-progress $URL
!test -d training || (echo "$SHA  aicup2026_go_training.tar.xz" | sha256sum -c && tar xf aicup2026_go_training.tar.xz)
!ls -la training | head -5
!du -sh training

## 3. 健全性檢查（約 30 秒）

測試涵蓋官方評分公式、棋盤提子不變量、切分互斥性、提交格式規則。

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
!python -m pytest -q 2>&1 | tail -3

## 4. 建手工特徵庫（約 25 分鐘 / 4 核）

用途有二：(a) 提供 `--quality-weight` 的輔助回歸目標；
(b) 跑出手工特徵地板，作為「深度模型有沒有贏過」的判斷基準。

若只想先跑通訓練，可以跳過這格並把訓練的 `--quality-weight` 設為 0。

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
import os, sys, subprocess

if os.path.exists('cache/features.npy'):
    print('特徵庫已存在，略過')
else:
    # 用 subprocess 而非 !command，避免依賴 IPython 在 if 區塊內的語法轉換
    subprocess.run([sys.executable, 'scripts/build_feature_store.py', '--nproc', '4'],
                   check=True)

## 5. 續訓設定

Kaggle 的 session 上限 12 小時，到點直接砍掉。所以：

- `--max-hours 11.5` 讓訓練自己提前收尾存檔
- `--save-every 500` 最多只損失 500 步
- checkpoint 寫到 `/kaggle/working/ckpt`，commit 後會保留成這個 notebook 的 output

**下一輪要接續訓練**：在右側 **Add Input → Notebook Output** 掛載這個 notebook 的上一個版本，
然後把下面的 `PREV` 改成掛載路徑，執行這一格把 checkpoint 複製過來，訓練指令加 `--resume`。

In [ ]:
import os, shutil, glob
OUT = '/kaggle/working/ckpt'
os.makedirs(OUT, exist_ok=True)

# 若掛載了上一版的 output，把 last.pt / best.pt 複製過來以便續訓
PREV = ''   # 例如 '/kaggle/input/aicup2026-go-train/ckpt'
if PREV and os.path.isdir(PREV):
    for f in ('last.pt', 'best.pt', 'history.json'):
        src = os.path.join(PREV, f)
        if os.path.exists(src):
            shutil.copy(src, OUT); print('已還原', f)
print('OUT =', OUT, '|', os.listdir(OUT) or '（空的，這是第一輪）')

## 6. 訓練

**第一次請先用小設定跑 2,000 步看曲線**，確認題目分數會往上再開大。
判斷標準不是 loss，是每 `--eval-every` 步印出的「題目分數」：

| 目標 | 分數 |
|---|---|
| 手工特徵地板（必須先贏過） | 0.3973 |
| 官方 CNN baseline | 0.4631 |

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
# --- 探路設定：約 20~30 分鐘，先確認會學 ---
!python scripts/train.py --device cuda \
    --width 64 --blocks 6 --embed-dim 256 --snapshots 2 \
    --p 16 --k 4 --steps 2000 \
    --eval-every 250 --eval-questions 800 \
    --save-every 250 --workers 4 \
    --balance-losses --quality-weight 0.3 \
    --max-hours 0.6 --out /kaggle/working/ckpt_probe

In [ ]:
%cd /kaggle/working/AI-CUP-2026-GO
# --- 正式設定：確認探路有效後再跑。加 --resume 可跨 session 接續 ---
!python scripts/train.py --device cuda \
    --width 128 --blocks 10 --embed-dim 512 --snapshots 4 \
    --p 32 --k 8 --steps 20000 \
    --eval-every 1000 --eval-questions 800 \
    --save-every 500 --workers 4 \
    --balance-losses --quality-weight 0.3 \
    --max-hours 11.5 --out /kaggle/working/ckpt

## 7. 訓練曲線與結論

In [ ]:
import json, os
import matplotlib.pyplot as plt

path = '/kaggle/working/ckpt/history.json'
if not os.path.exists(path):
    path = '/kaggle/working/ckpt_probe/history.json'
h = json.load(open(path))
steps = [r['step'] for r in h]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(steps, [r['score_expected'] for r in h], marker='o', ms=3, label='期望解碼')
ax[0].plot(steps, [r['score_argmax'] for r in h], marker='o', ms=3, alpha=.6, label='argmax')
ax[0].axhline(0.3973, ls='--', c='gray', label='手工特徵地板 0.3973')
ax[0].axhline(0.4631, ls='--', c='crimson', label='官方 baseline 0.4631')
ax[0].set_xlabel('step'); ax[0].set_ylabel('題目層級官方分數')
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

for k in ('loss_rank', 'loss_corn', 'loss_arc'):
    ax[1].plot(steps, [r[k] for r in h], marker='o', ms=3, label=k)
ax[1].set_xlabel('step'); ax[1].set_yscale('log'); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

best = max(r['score_expected'] for r in h)
print(f'最佳題目分數 {best:.4f}')
print('贏過手工特徵地板 (0.3973)：', '是' if best > 0.3973 else '否 —— 訓練量或實作有問題')
print('贏過官方 baseline (0.4631)：', '是' if best > 0.4631 else '否')

## 8. 把模型帶回去

`/kaggle/working/ckpt/best.pt` 會隨 commit 保留成 notebook output，可直接下載。

本機產生提交檔（競賽一的手工特徵版不需要 GPU，90 秒跑完 800 列）：

```bash
python3 scripts/predict.py --test-dir tests --out-dir submissions
```

## 常見問題

| 症狀 | 原因 / 處理 |
|---|---|
| `git clone` 或 `wget` 失敗 | Internet 沒開。Settings → Internet → On（需手機驗證） |
| 跑到一半整個 session 消失 | 用 Save & Run All (Commit) 而不是網頁互動模式 |
| 續訓時 checkpoint 不見了 | 要在 Add Input 掛載上一版的 Notebook Output，並設定第 5 格的 `PREV` |
| 題目分數在 0.1~0.25 徘徊不動 | 訓練量不足。本機 MPS 跑 3,000 步的小模型也只到 0.2548 |
| GPU 記憶體不足 | 調小 `--p` 與 `--k`（batch = p × k × snapshots） |